# Vignette of plotting hapROH output
In this vignette you will learn how to plot 

In [ ]:
import os as os
import numpy as np
import pandas as pd
import sys
import matplotlib.pyplot as plt
from matplotlib import gridspec

# 1) Various Plotting functions
General rule: savepath="PATH/FILE.pdf" controls where the resulting figure is saved to. If an empty string is given, the file will not be saved.
You can change the extension .pdf and .png, which will automatically use the according figure format. E.g. "./figure/fig.pdf" will save the figure as .pdf

### Plot the Posterior along one Chromosome. 
Needs the posterior to be saved,
(so in hapsb_ind use delete=False to keep it)

Set folder to the chromosome output you want to plot

In [ ]:
from hapsburg.figures.plot_posterior import plot_posterior_cm

In [ ]:
plot_posterior_cm(folder = "./Empirical/Eigenstrat/Example/I1178/chr4/", savepath="", 
                  empirical=True, m=1, cm_lim=[], groundtruth = False, min_cm=1,
                  readcount=False, figsize=(10,4), title="I0644, Chromosome 4")

### Zoom in
We now change the cm_lim parameter, which allows use to zoom in. It is a list of length two, and defines the region (in cM coordinates) to plot.

In [ ]:
plot_posterior_cm(folder = "./Empirical/Eigenstrat/Example/I1178/chr4/", savepath="", 
                  empirical=True, m=1, cm_lim=[0,50], groundtruth = False, min_cm=1,
                  readcount=False, figsize=(10,4), title="I0644, Chromosome 4")

### Plot Karyotype Plot with ROH
Plot inferred ROH on all chromosomes

In [ ]:
from hapsburg.figures.plot_individual_roh import plot_roh_individual

In [ ]:
plot_roh_individual(iid="I1178", folder="./Empirical/Eigenstrat/Example/", 
                    prefix_out="", min_cm=4, plot_bad=False, savepath="")  

### Wow, lots of ROH
That was the individuals with the most ROH in the sample

### Do Histogram of the Length Distribution and theoretical expectations

In [ ]:
from hapsburg.figures.plot_individual_roh import plot_pde_individual

In [ ]:
plot_pde_individual(iid=["KNC025"], min_cm=3, bw_cm=4, 
                    kde_plot=False, plotlim=[4, 60], prefix_out="",
                    savepath="", folder="/home/xiaowen_jia/Albania/KNC_HapROH/Output/")

### Plot the Lenght Distribution of ROH of multiple Individuals
If you pass on the iids as a list, it will load ROH of all of them and plot the joint histogram.
Expectations are rescaled accordingly. This is useful if you want to group several individuals with a similar profile and get a "average" curve with less noise.

In [ ]:
plot_pde_individual(iid=["I1178", "I0644"], min_cm=3, bw_cm=4, 
                    kde_plot=False, plotlim=[4, 60], prefix_out="",
                    savepath="", folder="./Empirical/Eigenstrat/Example/")

## 1.1) Plot summary over multiple Individuals

First plot only expectations, for instance as a legend.
You could choose which expectations in the method parameters

In [ ]:
from hapsburg.figures.plot_bars import plot_legend_only, plot_panel_row, prepare_dfs_plot

In [ ]:
plot_legend_only(savepath="", figsize=(4,8))

### Now plot an empirical dataset. 
prepare_dfs_plot prepared from the combined csv (splitting it up into a list of pandas dataframes, these dataframes will be plot in groups - you could also manually produce these lists)

plot_panel_row then plots these data frames  

In this simple usecase, the two individuals analyzed above (from two different populations) are plotted, together with a legend to the right.

In [ ]:
# For nice Arial Fonts
from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'   # Set the default
rcParams['font.sans-serif'] = ['Arial']  # Make sure to have the font installed (it is on cluster for Harald)

from hapsburg.PackagesSupport.roh_expectations import Expected_Roh

def create_cousins_roh(degrees=[1,2,3], 
                       bins=[[0.04,0.08],
                             [0.08,0.12],
                             [0.12,0.2],
                             [0.2,3.0]], bin_n=10000):
    """Create ROH sharing in list of bins (list of [begin,end]) 
    for Cousins of degree degrees [list]
    return sharing [len(degrees), len(bins)]"""
    e_roh = Expected_Roh()
    c_roh = np.zeros((len(degrees),len(bins))) # Container for results Cousins
    for i,c in enumerate(degrees):
        for j,b in enumerate(bins):
            m = c*2 + 4
            c_roh[i,j] = e_roh.exp_roh_len_in_bin_rel(l=b, m=m, comm_anc=4, bins=10000)
    return c_roh

#bins = [[0.04,0.08],[0.08,0.12],[0.12,3.00]]  # The bins I want to plot (eventually maybe do 12,16 as welll)
#cousins = [1, 2, 3]  # Which Cousins to Plot
def create_Ne_roh(Ns=[400, 800, 1600, 3200, 6400], 
                  bins=[[0.04,0.08],[0.08,0.12],[0.12,0.2],[0.2,3.0]], bin_n=10000):
    """Create ROH sharing in list of bins (list of [begin,end]) 
    for panmictic population sizes
    Ns: List of population sizes
    bins: Length Bins (in Morgan) to calculate expectations from
    return sharing [len(degrees), len(bins)]"""
    e_roh = Expected_Roh()
    n_roh = np.zeros((len(Ns),len(bins))) # Container for results Cousins
    for i,N in enumerate(Ns):
        for j,b in enumerate(bins):
            n_roh[i,j] = e_roh.exp_roh_len_in_bin_N(b, N=N, bins=bin_n)
    return n_roh

def std_Ne_roh(Ns=[400, 800, 1600, 3200, 6400], 
                  bins=[[0.04,0.08],[0.08,0.12],[0.12,0.2],[0.2,3.0]], bin_n=10000):
    """Create ROH sharing in list of bins (list of [begin,end]) 
    for panmictic population sizes
    Ns: List of population sizes
    bins: Length Bins (in Morgan) to calculate expectations from
    return sharing [len(degrees), len(bins)]"""
    e_roh = Expected_Roh()
    var_roh = np.zeros((len(Ns),len(bins))) # Container for results Cousins
    for i,N in enumerate(Ns):
        for j,b in enumerate(bins):
            var_roh[i,j] = e_roh.var_roh_len_in_bin_N(b, N=N, bins=bin_n)
    return np.sqrt(var_roh)  # Return Standard Deviation

######################################################################################
def plot_bar_ax(ax, y, bins=[], c=["#313695", "#abd9e9", "#fee090", "#d7191c"], x_ticks = [], 
                ec = "silver", fs_l=10, fs_y = 10, fs_x=8, fs_t=10, alpha=1.0,
                barWidth=0.95, ylim = [0,220], stds = [], 
                title="", ha_title="left", bold_title=False,
                yticks=False, ylabel="Sum Inferred ROH>4cM [cM]",
                legend=False, r_title=0, hlines=[]):
    """Plot bars of ROH on Axis.
    ax: Where to Plot on
    y: Array of ROH to plot: [n Inds, k Legnth Bins]
    c: Which colors to plot
    bins: List of Bins (needed for legend - plotted if len()>0)
    yticks: Whether to plot Y tick Labels
    legend: Whether to plot Legend
    fs_x, fs_y: Fontsize on the x and yLabels
    r_title: Rotation of the title
    hlines: List where to plot hlines"""
    x = np.arange(len(y))

    for i in range(len(y[0,:])): # From last to first (For Legend)
        b = np.sum(y[:,:i], axis=1)
        ax.bar(x, y[:,i], bottom=b, color=c[i], edgecolor=ec, width=barWidth, 
                   label=f"{bins[i,0]}-{bins[i,1]} cM", alpha=alpha, zorder=4)
        if len(stds)>0 and i>0: # Plot some standard deviations.
            ax1.errorbar(r, b, yerr=stds[:,i], fmt='none', linewidth=2, color="k")
    
    if len(hlines)>0:
        for y in hlines:
            ax.axhline(y=y, zorder=0, linestyle="--", color="gray", lw=0.5)     
    
    if legend:
        ax.legend(fontsize=fs_l, loc="upper right", title="Sum ROH in")

    
    ### Do the xtricks
    ax.set_xticks(x)
    
    if len(x_ticks)>0:
        ax.set_xticklabels(x_ticks, fontsize=fs_x, rotation=270)
    else:
        ax.set_xticklabels([])
    if not yticks:
        ax.axes.yaxis.set_ticks([])
        #ax.set_yticklabels(y_ticks, fontsize=fs_x)
    else:
        ax.tick_params(axis='y', labelsize=fs_y)
    fw = 'normal'
    if bold_title:
        fw = "bold"
    ax.set_ylabel(ylabel, fontsize=fs_y)
    ax.set_ylim(ylim)
    ax.set_xlim(x[0] - 0.7*barWidth, x[-1] + 0.7*barWidth)
        
    if len(title)>0:
        ax.set_title(title, fontsize=fs_t, rotation=r_title, 
                     horizontalalignment=ha_title, fontweight=fw)
        
def plot_close_kin(ax, y, y_plot=100,cutoffs=[50,100], c="r", ec="k", lw=1,
                   ss=[10,14], m_cs=["v", "s"], bin_idx=-1):
    """Plot Symbols for close kin at height y_symbol
    ss: Sizes of markers [List]
    m_cs: marker_colors [List]
    cutoffs: cutoffs [List]
    """
    if len(y)==0:  # Move out if nothing to plot
        return
    x = np.arange(len(y))
    y0 = y[:,bin_idx]
    
    cutoffs_t = cutoffs + [4000]
    
    for i in range(len(cutoffs)):
        idx = (y0>cutoffs_t[i]) & (y0<cutoffs_t[i+1])
        if np.sum(idx)==0:
            continue
        ax.scatter(x[idx], [y_plot]*np.sum(idx), c=c, ec=ec, lw=lw, 
                   marker=m_cs[i], s=ss[i], zorder=5)

In [ ]:
def plot_panel_row(plot_dfs, wspace=0.05, hspace=0.01, figsize=(24,3.5), savepath="", 
                   x_labels=[], ylabel="Sum Inferred ROH>4cM [cM]", c=["#313695", "#abd9e9", "#fee090", "#d7191c"], 
                   ylim = [0,250], r_title = 90, bolds=[],
                   fs_l=10, fs_y = 10, fs_x=8, fs_t=10, ha_title="left", hspace_leg=1,
                   leg_pos = -2, show=True, title_col="clst", titles=[], hlines=[],
                   cols = ['sum_roh>4', 'sum_roh>8', 'sum_roh>12', 'sum_roh>20'],
                   bins = [[0.04, 0.08], [0.08, 0.12], [0.12, 0.2], [0.2, 3.0]],
                   degrees=[1, 2, 3], Ns=[400, 800, 1600, 3200, 6400],
                   ticks_c=["1st Cousin", "2nd Cousin", "3rd Cousin"],
                   cutoffs=[], ec="k", lw=1, ss=[40, 50], 
                   m_cs=["v", "s"], sym_ofst=-10, bin_idx=-1, dpi=600):
    """
    Plot row of ROH bin plots from plot_dfs (each df one panel)
    and save both PNG and SVG if savepath is provided.
    """
    bins_cM = (np.array(bins) * 100).astype("int")
    n_plots0 = len(plot_dfs)  # The original plots
    n_plots = len(plot_dfs) + 1  # Add 1 for the empty plot
    width_ratios = [len(df) for df in plot_dfs]
    width_ratios += [hspace_leg]

    # Space for legends
    if len(degrees) > 0: 
        n_plots += 1
        width_ratios.append(len(degrees))
    if len(Ns) > 0: 	    
        n_plots += 1
        width_ratios.append(len(Ns))
        
    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs = gridspec.GridSpec(1, n_plots, width_ratios=width_ratios, figure=fig)
    gs.update(wspace=wspace, hspace=hspace)  # set the spacing between axes

    # Plot each cluster
    for i, df in enumerate(plot_dfs):   
        legend = (i == (len(plot_dfs) + leg_pos))
        ax = plt.subplot(gs[i])  # Extract the Sub Plot to Plot onto

        obs_roh = df[cols].values
        # Calculate the value in the Bins
        for j in range(len(cols)-1):
            obs_roh[:, j] = obs_roh[:, j] - obs_roh[:, j+1]

        # X-ticks
        if x_labels is False:
            x_ticks0 = []
        elif len(x_labels) > 0:
            x_ticks0 = x_labels[i]
        else:
            x_ticks0 = df["iid"].values

        # Title
        title = titles[i] if len(titles) > 0 else df[title_col].values[0]
        bold = bolds[i] if len(bolds) > 0 else False

        # Y-axis label only on the first subplot
        ylabel1 = ylabel if i == 0 else ""
        yticks = (i == 0)

        plot_bar_ax(ax, obs_roh, bins_cM, legend=legend, r_title=r_title, c=c,
                    x_ticks=x_ticks0, title=title, ha_title=ha_title, bold_title=bold,
                    ylim=ylim, hlines=hlines, ylabel=ylabel1, yticks=yticks,
                    fs_l=fs_l, fs_y=fs_y, fs_x=fs_x, fs_t=fs_t)

        # Close kin markers
        if len(cutoffs) > 0:
            c_marker = np.array(c)[bin_idx]
            plot_close_kin(ax, obs_roh, y_plot=ylim[1] + sym_ofst, cutoffs=cutoffs, 
                           c=c_marker, ec=ec, lw=lw, ss=ss, m_cs=m_cs, bin_idx=bin_idx)

    # Placeholder axis
    ax_none = plt.subplot(gs[n_plots0])
    ax_none.set_visible(False)

    # Cousin Legend
    if len(degrees) > 0:
        c_roh = create_cousins_roh(degrees=degrees, bins=bins)
        ax_c = plt.subplot(gs[n_plots0 + 1])
        plot_bar_ax(ax_c, c_roh * 100, bins_cM, legend=False, ylim=ylim, c=c, 
                    hlines=hlines, x_ticks=ticks_c, yticks=False, ylabel="",
                    fs_l=fs_l, fs_y=fs_y, fs_x=fs_x, fs_t=fs_t,
                    title="Recent Loops", r_title=r_title)
    
    # Small Population Legend
    if len(Ns) > 0:
        ns_roh = create_Ne_roh(Ns=Ns, bins=bins)
        pos_leg_N = n_plots0 + 1 + (len(degrees) > 0)  # Python indexing
        ax_N = plt.subplot(gs[pos_leg_N])
        ticks_N = [f"2N={i}" for i in Ns]
        plot_bar_ax(ax_N, ns_roh * 100, bins_cM, legend=True, ylim=ylim, c=c,
                    yticks=False, ylabel="", hlines=hlines, x_ticks=ticks_N,
                    title="Small Pop. Size", r_title=r_title,
                    fs_l=fs_l, fs_y=fs_y, fs_x=fs_x, fs_t=fs_t)

    # Save outputs
    if len(savepath) > 0:
        # Save PNG
        plt.savefig(savepath, bbox_inches='tight', pad_inches=0, dpi=300)
        print(f"Saved figure to {savepath}")
        
        # Save SVG
        svg_path = savepath.rsplit('.', 1)[0] + ".svg"
        plt.savefig(svg_path, format='svg', bbox_inches='tight', pad_inches=0)
        print(f"Saved SVG figure to {svg_path}")
        
    if show:
        plt.show()
    return


In [ ]:
df1 = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/HapROH/0925/KNC_0925_roh05.csv", sep='\t')
plot_dfs, cols = prepare_dfs_plot(df1, cms=[4, 8, 12, 20])

# ./figures_test/freilich20_bars.pdf
plot_panel_row(plot_dfs, wspace=0.1, r_title=0, leg_pos=5, 
               ylim=[0,250], figsize=(100,15), savepath="KNC_ROH_0226", fs_l=50, fs_y = 50, fs_x=25, fs_t=60)

In [ ]:
def plot_legend_only(figsize=(7,6), wspace=0.05, hspace=0.01, savepath="", hlines=[],r_title=0,
                     fs_l=10, fs_y = 10, fs_x=8, fs_t=10, c=["#313695", "#abd9e9", "#fee090", "#d7191c"],
                     bins = [[0.04, 0.08], [0.08, 0.12], [0.12, 0.2], [0.2,3.0]], ha_title="center",
                     degrees=[1, 2, 3], Ns=[400, 800, 1600, 3200, 6400], ylim = [0,250],
                     x_ticks_c = ["1st Cousin", "2nd Cousin", "3rd Cousin"], title_c = "Recent Loops",
                     title_N="Small Pop. Size", y_label="Expected Sum ROH>4cM [cM]"
                     ):
    """Plot Inbreeding from recent Cousins as well as small pop size.
    bins: list of length bins to plot [[a1,a2],...[z1,z2]]
    Ns: What population sizes to plot in barplot [list]
    degrees: What degrees of Cousins to plot. [list]"""
    width_ratios = [len(degrees), len(Ns)]
    bins_cM=(np.array(bins)*100).astype("int")
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(1, 2, width_ratios=width_ratios, figure=fig)
    ax_cousin = plt.subplot(gs[0])    # The left subplot (Timeline)
    ax_Ne = plt.subplot(gs[1])
    gs.update(wspace=wspace, hspace=hspace) # set the spacing between axes

    ### Calcualte Expectations Cousins:
    c_roh = create_cousins_roh(degrees = degrees, bins = bins) # in Morgan
    
    ### Calculate Expectations Ne:
    sum_roh = create_Ne_roh(Ns=Ns, bins = bins) # in Morgan
    
    plot_bar_ax(ax_cousin, c_roh*100, bins_cM, legend=False, r_title=r_title, 
                fs_l = fs_l, fs_y = fs_y, fs_x = fs_x, fs_t = fs_t, c = c,
                ylim = ylim, ylabel = "", yticks=False, x_ticks = x_ticks_c,
                hlines=hlines, title=title_c, ha_title=ha_title,
                )

    ticks_N = [f"2N={i}" for i in Ns]
    plot_bar_ax(ax_Ne, sum_roh*100, bins_cM, legend=True, r_title=r_title, 
                fs_l=fs_l, fs_y = fs_y, fs_x = fs_x, fs_t = fs_t, c = c, 
                x_ticks = ticks_N, ylim = ylim, 
                ylabel="", yticks=False,
                hlines=hlines, title=title_N, ha_title=ha_title)
            
    if len(savepath)>0:
        plt.savefig(savepath, bbox_inches = 'tight', pad_inches = 0, dpi=600)
        print(f"Saved figure to {savepath}") 

        # Save SVG
        svg_path = savepath.rsplit('.', 1)[0] + ".svg"
        plt.savefig(svg_path, format='svg', bbox_inches='tight', pad_inches=0)
        print(f"Saved SVG figure to {svg_path}")
        
    plt.show()

In [ ]:
plot_legend_only(savepath="KNC_legend", figsize=(6,15),fs_x=25,r_title=90, fs_t=50)

Looks like we detected one highly inbred individual!

# Advanced Plotting: Using a map and fitting of ROH over time
Warning: requires basemap package, which can be tricky to install. Also needs resutls file with additional meta data (.csv) with latitude, longitude and age for each sample.
The big datatable in the vignette folder is one such example, which you can use to explore, or as a template for visualizing your own results.
You can also use a color column in that meta file - this determines in which color each individuals is plot

In [ ]:
import pandas as pd
from hapsburg.figures.plot_timelines import plot_map_time, extract_pop, prep_label

### Example using the global results .csv 
[global results will be released with publication]
For detailled explanation of all the possible parameters check the doc string of the method.
(they have reasonable default values, but I show them here so you get a sense what can be done in principle).

You can change what map is plotted by updating the corners in the parameter crs_m (in the form [lat0, lat1, lon0, lon1]). You probably have to play around with the with_ratios then to get nicely matching squares.

If you set std_band to 0 the GP fitting is not plotted - that might be useful when you have few individuals. Otherwise lght_s sets the lower and upper bound of the "covariance" kernels (bigger values: points affect more distant times)

In [ ]:
pop = "Balkans"
df1 = pd.read_csv("./Empirical/roh_all_inds_final_v42.csv", sep="\t")
### Now we subset to all individuals having the "region" column defined above. There is a function for that,
# but feel free to use and subsetting to individuals (i.e. rows) you prefer/need
df_t = extract_pop(df1, age_range=[0,12200], pop=pop) # Subsetting to dataframe to plot
label = prep_label(df_t, pop) # pop: What to plot in legend. Automatically produces label. You can also just write it.

plot_map_time(df_t, figsize=(7.2 , 2.0), crs_m=[28, 63, -11, 38], 
              width_ratios=(8, 20), height_ratios=[15, 1], hspace=0.06, wspace=0.015,
              s_tl=20, ec="k", lw=0.18, x_lim_tl=[-500, 12200], vrange_m=[0,12200], 
              y_lim_tl=[0,120], fsl=8, fs=9, fs_leg=9, leg_loc_tl="", title_tl="",
              map_title=label, title_loc=(0.2,0.01), cm=4, cm1=8, frac=0, std_band=1.92, 
              lgth_s=[1500,1500], bottomrow=True, rightcol=True, lw_fit=1.5, 
              ticks=[83.82, 21.84], tick_l=[f"$2N_e=500$", f"$2N_e=2000$"], 
              width_t=0.6, length_t=2, lbl_pad_time=10, lbl_pad_age=0, xl_pad=1.5, yl_pad=1, 
              widths=800, alpha_vio=0.4, savepath="")